# Week06 Part 1 — 임베딩과 유사도 검색 (무료·API 키 불필요) [실습용]

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itlects/2026-1_ai-convergence-apply/blob/main/Week06/Part1/Week06_Part1_Embedding_Similarity_%EC%8B%A4%EC%8A%B5%EC%9A%A9.ipynb)

**목표:** 문장을 임베딩 벡터로 바꾸고, 코사인 유사도로 의미가 가까운 문장을 찾는다. TF-IDF(글자 일치)와 임베딩(의미 일치)을 비교하고, 문서를 청크로 나눠 출처가 있는 검색기를 만든다.

**학습 방법:** Notion 워크북의 실습 번호를 따라 `TODO`가 있는 셀을 직접 완성하고, 위에서부터 순서대로 실행하세요. 막히면 같은 폴더의 **완성본**을 참고합니다.

## 실습 0. 실행 환경 준비
- Colab 메뉴 **런타임 → 런타임 유형 변경**에서 CPU 그대로 사용해도 됩니다(GPU 불필요).
- 첫 실행 시 한국어 임베딩 모델(`jhgan/ko-sroberta-multitask`, 약 450MB)을 내려받으므로 1~2분 걸립니다.

In [ ]:
import importlib.util
if importlib.util.find_spec("sentence_transformers") is None:
    !pip install -q sentence-transformers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("jhgan/ko-sroberta-multitask")
print("모델 준비 완료")

## 실습 1. 문장을 임베딩 벡터로 바꾸기
임베딩(embedding)은 문장의 **의미**를 숫자 목록(벡터)으로 표현한 것입니다.
`model.encode()`에 문장 목록을 넣으면 문장마다 하나의 벡터가 나옵니다.

In [ ]:
sentences = [
    "RAG는 검색 증강 생성이다.",
    "등록금 납부 기간이 언제인가요?",
    "학비는 언제까지 내야 하나요?",
]
# TODO 1: model.encode()로 sentences를 벡터로 바꾸세요.
vectors = ...

print("벡터 배열 모양:", vectors.shape)          # (문장 수, 벡터 차원)
print("첫 문장 벡터 앞 5개 값:", np.round(vectors[0][:5], 3))

**확인:** 벡터 차원은 몇인가요? 문장 길이가 달라도 차원이 같은 이유를 한 문장으로 적어 보세요.

## 실습 2. 코사인 유사도 직접 계산하기
두 벡터 a, b의 코사인 유사도 = (a·b) / (|a|×|b|)

값이 1에 가까울수록 같은 방향(비슷한 의미), 0에 가까울수록 관련 없음입니다.

In [ ]:
def cosine(a, b):
    # TODO 2: 코사인 유사도 공식을 완성하세요. (np.dot, np.linalg.norm 사용)
    return ...

print("문장2 vs 문장3 (같은 의미, 다른 단어):", round(cosine(vectors[1], vectors[2]), 3))
print("문장1 vs 문장2 (다른 주제)          :", round(cosine(vectors[0], vectors[1]), 3))

# sklearn 결과와 같은지 확인
print("sklearn으로 계산한 문장2 vs 문장3   :", round(float(cosine_similarity([vectors[1]], [vectors[2]])[0][0]), 3))

## 실습 3. TF-IDF(글자 일치) vs 임베딩(의미 일치) 비교
TF-IDF는 **같은 글자/단어가 얼마나 겹치는지**로 점수를 매기고, 임베딩은 **의미**로 점수를 매깁니다.
단어는 다르지만 뜻이 같은 문장 쌍에서 두 방식의 점수를 비교합니다.

In [ ]:
pairs = [
    ("등록금 납부 기간이 언제인가요?", "학비는 언제까지 내야 하나요?"),
    ("실습실 컴퓨터가 고장 났어요", "랩실 PC가 작동하지 않습니다"),
    ("장학금 신청 방법 알려주세요", "장학금 신청서는 어디서 받나요?"),
    ("장학금 신청 방법 알려주세요", "실습실 개방 시간이 궁금해요"),
    # TODO 3: 뜻은 같지만 단어가 다른 문장 쌍을 하나 추가하세요.
]

all_text = [s for p in pairs for s in p]
tfidf = TfidfVectorizer(analyzer="char", ngram_range=(2, 3)).fit(all_text)

rows = []
for a, b in pairs:
    tfidf_score = cosine_similarity(tfidf.transform([a]), tfidf.transform([b]))[0][0]
    ea, eb = model.encode([a, b])
    rows.append({"문장 A": a, "문장 B": b,
                 "TF-IDF": round(float(tfidf_score), 3),
                 "임베딩": round(cosine(ea, eb), 3)})
pd.DataFrame(rows)

**확인:** 단어가 거의 겹치지 않는 1·2번 쌍에서 어떤 방식의 점수가 더 높은가요? 이것이 RAG 검색에 임베딩을 쓰는 이유입니다.

## 실습 4. 유사도 행렬 그리기
여러 문장끼리의 유사도를 한 번에 계산해 색으로 표시합니다. (그래프 글자 깨짐을 피하려고 축은 S1~S6 번호로 표시합니다.)

In [ ]:
samples = [
    "수강신청은 언제 하나요?",
    "수강 정정 기간을 알려주세요",
    "국가장학금 신청 기간이 궁금해요",
    "장학금은 언제 들어오나요?",
    "실습실 프린터가 안 돼요",
    "3D 프린터 사용 예약 방법",
]
for i, s in enumerate(samples, 1):
    print(f"S{i}: {s}")

emb = model.encode(samples)
sim = cosine_similarity(emb)

labels = [f"S{i}" for i in range(1, len(samples) + 1)]
plt.figure(figsize=(5, 4))
plt.imshow(sim, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="cosine similarity")
plt.xticks(range(len(labels)), labels)
plt.yticks(range(len(labels)), labels)
for i in range(len(samples)):
    for j in range(len(samples)):
        plt.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Sentence similarity matrix")
plt.show()

**확인:** 같은 주제(수강/장학/실습실)끼리 점수가 높게 묶이나요? 이 성질을 Part 2에서 **질문 분류**에 사용합니다.

## 실습 5. 문서를 청크로 나누기 (Chunking)
긴 문서를 통째로 비교하면 질문과 관계없는 내용까지 섞입니다. 그래서 **문장 몇 개 단위의 조각(청크)**으로 나누고, 각 청크에 **출처 번호**를 붙입니다.

In [ ]:
guide = (
    "전자공학과 실습실은 평일 오전 9시부터 오후 9시까지 개방한다. "
    "주말과 공휴일에는 사전 예약한 학생만 실습실을 사용할 수 있다. "
    "실습실 장비가 고장 나면 조교실에 고장 신고서를 제출한다. "
    "3D 프린터는 학과 홈페이지에서 하루 전까지 예약해야 한다. "
    "수강신청은 학기 시작 2주 전에 학사 시스템에서 진행한다. "
    "수강 정정은 개강 후 첫 주에만 가능하다. "
    "국가장학금은 한국장학재단 홈페이지에서 학기마다 신청한다. "
    "교내 성적장학금은 직전 학기 성적을 기준으로 자동 선발된다. "
    "정보처리기능사 자격증 특강은 방학 중에 학과에서 운영한다. "
    "취업 상담은 학과 사무실에서 매주 수요일에 예약할 수 있다."
)

def split_sentences(text):
    parts = [p.strip() for p in text.split(". ") if p.strip()]
    return [p if p.endswith(".") else p + "." for p in parts]

def make_chunks(text, source, size=2, overlap=1):
    sents = split_sentences(text)
    step = size - overlap
    chunks = []
    for start in range(0, len(sents), step):
        piece = sents[start:start + size]
        if not piece:
            break
        chunks.append({"id": f"{source}#{len(chunks)}", "text": " ".join(piece)})
        if start + size >= len(sents):
            break
    return chunks

# TODO 5: size=2, overlap=1로 청크를 만드세요. 실행 후 size=3, overlap=0으로도 바꿔 보세요.
chunks = make_chunks(guide, source="학과안내", size=..., overlap=...)
print("문장 수:", len(split_sentences(guide)), "/ 청크 수:", len(chunks))
for c in chunks[:3]:
    print(c["id"], "→", c["text"])

**확인:** `overlap=1`이면 이웃한 청크가 문장 1개를 공유합니다. 겹침이 있으면 어떤 장점이 있을까요?

## 실습 6. 출처가 있는 의미 검색 + 임계값
질문을 임베딩해 모든 청크와 유사도를 계산하고, 점수가 높은 청크를 **출처와 함께** 보여줍니다.
가장 높은 점수도 `threshold`(임계값)보다 낮으면 **"문서에서 근거를 찾을 수 없음"**으로 처리합니다.

In [ ]:
chunk_vecs = model.encode([c["text"] for c in chunks])

def search(question, k=3, threshold=0.45):
    q = model.encode([question])
    # TODO 6: 질문 벡터 q와 chunk_vecs의 코사인 유사도를 계산하세요.
    scores = ...
    order = scores.argsort()[::-1][:k]
    hits = [(chunks[i]["id"], round(float(scores[i]), 3), chunks[i]["text"]) for i in order]
    found = hits[0][1] >= threshold
    return found, hits

for question in ["실습실은 몇 시까지 열어요?",
                 "장학금 신청은 어디서 해?",
                 "이번 학기 중간고사 날짜는?"]:
    found, hits = search(question)
    print(f"\n질문: {question}")
    if not found:
        print(f"  → 문서에서 근거를 찾을 수 없음 (최고 점수 {hits[0][1]})")
    for cid, score, text in hits:
        print(f"  [{cid}] {score}  {text}")

**확인:** 문서에 없는 '중간고사 날짜' 질문의 최고 점수는 얼마인가요? `threshold`를 0.3, 0.45, 0.6으로 바꾸면 결과가 어떻게 달라지는지 결과표에 기록하세요.

## 실습 7. (도전) 나만의 문서 추가
자신이 고른 안내문·장비 매뉴얼 **5문장 이상**을 `my_text`에 넣고 새 출처로 추가한 뒤, 질문 3개로 검색해 보세요.

In [ ]:
my_text = (
    "여기에 나만의 문서를 5문장 이상 입력하세요. "
    "문장은 마침표와 공백으로 구분합니다. "
    "예: 오실로스코프는 사용 전 프로브 보정을 한다. "
    "측정이 끝나면 전원을 끄고 프로브를 정리한다. "
    "고장 시 조교에게 즉시 알린다."
)
chunks += make_chunks(my_text, source="나의문서", size=2, overlap=1)
chunk_vecs = model.encode([c["text"] for c in chunks])
print("전체 청크 수:", len(chunks))

for question in ["프로브 보정은 언제 하나요?", "질문 2를 입력하세요", "질문 3을 입력하세요"]:
    found, hits = search(question)
    print(f"\n질문: {question} / 근거 있음: {found}")
    for cid, score, text in hits:
        print(f"  [{cid}] {score}  {text}")

## 결과 기록표 (노션 결과표에 옮겨 적기)
| 질문 | 1순위 출처(청크 ID) | 유사도 | 근거 있음/없음 | 정답 여부 | 관찰 |
|---|---|---|---|---|---|
| 실습실은 몇 시까지 열어요? |  |  |  |  |  |
| 장학금 신청은 어디서 해? |  |  |  |  |  |
| 이번 학기 중간고사 날짜는? |  |  |  |  |  |

**참고:** TF-IDF는 글자 통계 기반 검색, 임베딩은 의미 기반 검색입니다. 다음 Part 2에서는 이 임베딩 검색을 **질문 카테고리 분류**에 적용합니다.